# Developing data simulation

The objective of this notebook is helping to develop step by step the simulation from `R` into `python`

The first objective will be the creation of a simulation possibility based only on continuous features. No categorical will be considered.

In [1]:
import torch

In [2]:
means = torch.tensor([3.5,-3.5], dtype=torch.float64)
covs = torch.tensor([[1,-0.5],[-0.5,1]], dtype=torch.float64)
mvn = torch.distributions.MultivariateNormal(means, covariance_matrix=covs)

In a 2 dimensional multivariate distribution, we have a vector $X$ of two RV
$$
X = \begin{pmatrix} X_1 \\ X_2 \end{pmatrix}
$$
Mean and VCOV matrix are given by
$$
\mu = \begin{pmatrix} 3.5 \\ -3.5 \end{pmatrix}, \; 
\Sigma = \begin{pmatrix}
1 & -\frac{1}{2} \\
-\frac{1}{2} & 1
\end{pmatrix}
$$

This means when sampling we will only get positive numbers for $X_1$ realizations and negative numbers for $X_2$ realizations

In [3]:
sample_2d = mvn.sample((3,))
print(sample_2d.shape)
sample_2d

torch.Size([3, 2])


tensor([[ 4.0739, -2.5579],
        [ 3.5667, -2.2065],
        [ 4.5279, -4.0313]], dtype=torch.float64)

So in a sample size of `(n,)`, the shape will be `[n, k]` with `k` the number of random variables composing $X$.

With $x^{(r)}_j$ the $j$-th sampled realization of $X_r$, the output looks like
$$
\texttt{sample} = \begin{pmatrix}
    x^{(1)}_1 & \cdots & x^{(k)}_1 \\
    \vdots & \ddots & \vdots \\
    x^{(1)}_n & \cdots & x^{(k)}_n
\end{pmatrix} \in \mathbb{R}^{n \times k}
$$


## A note on Gaussian Mixtures

A Gaussian mixture is defined by a categorical random variable and $K \in \mathbb{N}$ components. Formally: with
$$
Z \sim \text{Categorical}\left(\pi_1, \ldots, \pi_K \right), \; \sum_{k=1}^K \pi_k = 1
$$
whereby
$$
\mathbb{P}(Z = k) = \pi_k, k \in \{1, \ldots, K \}
$$

(which of course implies $\pi \in [0,1]$), $X$ is called a gaussian mixture of $K$ components each with $f$ elements if it's distributed as
$$
X | \{Z=k\} \sim \mathcal{N}\left(\mu_k, \Sigma_k \right)
$$

whereby $\mu_k = (\mu_{k,1}, \ldots, \mu_{k,f})$


An alternative (a bit less formal) formulation that makes very clear the implementation details is
$$
X = \mu_Z + \Sigma^{1/2}_Z \epsilon, \epsilon \sim \mathcal{N}(0, I_f)
$$

whereby $\Sigma_k = \Sigma_k^{1/2} {\Sigma_k^{1/2}}^\top$. A natural choise for $\Sigma_k^{1/2}$ is the cholesky decomposition of $\Sigma_k$.

A helpful realization is to see that the pdf of $X$ is given by
$$
p(x) = \sum_{k=1}^K \pi_k \phi_{\mu_k, \Sigma_k}(x)
$$
which also explains the 

## Notes on the R-Algorithm

### Calls:

First call is made for the init_population, by

```r
res <- genCreditData(
  #################################### DIMENSIONALITY
  n                = init_sample,  # - sample size = 100
  bad_ratio        = bad_ratio,    # - BAD ratio (if all D = 0) = 0.7
  k_con            = num_feats,    # - no. continuous features = 2
  k_cat            = 0,            # - no. categorical features
  k_bin            = 0,            # - no. binary features
  k_noise          = num_noise,    # - no. white-noise features = 0
  #################################### CONTINUOUS FEATURES
  con_nonlinear    = 0.0,       # - share of nonlinear transformations
  con_mean_bad_dif = mean_dif,  # - mean difference between classes = c(2, 1)
  con_var_bad_dif  = var_dif,   # - share of var/covar difference between classes = 0.5
  con_noise_var    = noise_var, # - variance of noise = 0
  covars           = covars,    # - variance-covariance matrices = list(matrix(c(1,  0.2,  0.2, 1), nrow = 2), matrix(c(1, -0.2, -0.2, 1), nrow = 2))
  #################################### MIXTURE OF GAUSSIANS
  mixture      = mixture,       # - mixture of two Gaussians = FALSE
  mix_mean_dif = mix_mean_dif,  # - mean difference between components  = 5
  mix_var_dif  = mix_var_dif,   # - share of var/covar difference between components = 0
  #################################### OTHER PARAMETERS
  seed             = seed,   # - random seed
  verbose          = F,      # - displaying feedback
  encode_factors   = F)      # - encoding of categorical features
```

`matrix(c(1,  0.2,  0.2, 1)` liefert
$$
\begin{pmatrix}
    1 & 0.2 \\
    0.2 & 1
\end{pmatrix}
$$

The next call is done this way:
```r
new_applicants <- genCreditData(n = sample_size, replicate = res, seed = seed + g)$data
```

Which is a quick way to replicate the arguments of the last call, based on this object (all names are set as objects - which are "`names`" in R)
```r
list(n                = n,
     k_cat            = k_cat,
     k_bin            = k_bin,
     k_noise          = k_noise,
     bad_ratio        = bad_ratio,
     con_mean_bad_dif = con_mean_bad_dif,
     con_var_bad_dif  = con_var_bad_dif,
     con_nonlinear    = con_nonlinear,
     con_noise_var    = con_noise_var,
     mixture          = mixture,
     mix_mean_dif     = mix_mean_dif,
     mix_var_dif      = mix_var_dif,
     cat_levels       = cat_levels,
     cat_var_share    = cat_var_share,
     cat_nonlinear    = cat_nonlinear,
     cat_noise_var    = cat_noise_var,
     bin_prob         = bin_prob,
     bin_mean_bad_dif = bin_mean_bad_dif,
     bin_bad_ratio    = bin_bad_ratio,
     bin_mean_con_dif = bin_mean_con_dif,
     bin_var_bad_dif  = bin_var_bad_dif,
     bin_noise_var    = bin_noise_var,
     encode_factors   = encode_factors,
     verbose          = verbose,
     seed             = seed)
```

however n and seed are not "taken over" - the rest they are

### Relevant parts of the algorithm

1. Set `combo_bad_ratio <- 0.7` and `combo_count <- n`
2. Compute "good" and "bad" sample sizes, as $n \cdot 0.7$ und $n \cdot (1-0.7)$ correspondingly. Ties are solved by 50/50 chance of doing good = n - bad or bad = n - good.
3. Set `mu_1 = c(0,0)` and therefore `mu_2 = c(1,2)=con_mean_bad_dif` (see line 249)

In [ ]:
from typing import List, Optional, Tuple, Union
import torch
def random_vcov_matrix(
        f: int,
        generator: Optional[torch.Generator] = None,
        var_range: Tuple[float, float] = (0.0, 1.0),
        prefer_normal_base_sampling: bool = True,
        device: torch.device = torch.get_default_device(),
        dtype: torch.dtype = torch.get_default_dtype(),
        eps: float = 1e-6
    ) -> torch.Tensor:
    """Generate a random positive definite covariance matrix.

    The covariance matrix is produced by:

    1. Generating a base sampling of a matrix :math:`A` (normal or uniform).
    2. Creating a correlation matrix via cosine similarity between the row vectors
       of the base sampling:

       .. math::

          C = (c_{i,j}) = \\left( \\frac{\\langle A_i, A_j \\rangle}{\\|A_i\\| \\|A_j\\|} \\right)

    3. Sampling a variance vector uniformly in ``var_range``, which is used to
       rescale the correlation matrix.
    4. Ensuring positive definiteness via addition of a small diagonal
       perturbation ``eps``.
    5. Make sure exact symmetry, so that rounding point instability does not
       lead to unsymmetric results. 

    Args:
        f (int): Dimension of the covariance matrix.
        generator (torch.Generator, optional): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances. Defaults to (0.0, 1.0).
        prefer_normal_base_sampling (bool, optional): If True, use normal distribution for base sampling.
            If False, use uniform distribution. Defaults to True.
        device (torch.device, optional): Device on which to allocate the tensor.
            Defaults to ``torch.get_default_device()``.
        dtype (torch.dtype, optional): Data type of the returned tensor.
            Defaults to ``torch.get_default_dtype()``.
        eps (float, optional): Small positive value added to the diagonal to ensure positive definiteness.
            Defaults to 1e-6.

    Returns:
        torch.Tensor: A symmetric, positive definite covariance matrix of shape ``(k, k)``.

    Raises:
        ValueError: If ``var_range`` is not a valid (min, max) tuple.

    Example:
        >>> g = torch.Generator().manual_seed(42)
        >>> cov = random_vcov_matrix(4, generator=g)
        >>> cov.shape
        torch.Size([4, 4])
    """

    # Step 1: Generate base sampling
    if prefer_normal_base_sampling:
        A = torch.randn((f, f), generator=generator, dtype=dtype, device=device) # random normal matrix, sparser correlations for high k
    else:
        A = 2*torch.rand((f,f), generator=generator, dtype = dtype, device=device) - 1 # random uniform matrix, correlations closer to 0, the higher k

    # Step 2: Define correlation matrix from base sampling
    Q = A @ A.T # Make sure of symmetry while using full randomness
    D = torch.sqrt(torch.diag(Q)) # Help vector for normalization
    corr_mat = Q / torch.outer(D, D) # corr_mat[i, j] = cosine_similarity(A[i], A[j]), so range [-1, 1] guaranteed

    # Step 3: Rescale corr_mat with sampled variances
    ## Variance sampling from uniform distribution
    variances = torch.rand(f, generator=generator, dtype = dtype, device=device) * (var_range[1] - var_range[0]) + var_range[0]
    ## Rescaling via outer prouct of standard deviations
    stds = torch.sqrt(variances)
    norm_factors_pearson_corr = torch.outer(stds, stds) # guaranteed to be symmetric, denominators of pearson correlation
    vcov = corr_mat * norm_factors_pearson_corr

    # Step 4: Avoid semi positive definitness of the matrix
    vcov = vcov + eps * torch.eye(f, device=device, dtype=dtype)

    # Step 5: Ensure **exact** symmetry without compromising randomness
    i, j = torch.tril_indices(f, f, offset=-1)
    vcov[i, j] = vcov[j, i]

    
    return vcov

def eigen_decomp_proj_to_pd(
    mat: torch.Tensor,
    eps: float = 1e-6,
    ensure_symmetry: bool = False
) -> torch.Tensor:
    """Project a matrix onto the positive definite (PD) cone via eigen-decomposition.

    The procedure ensures the output is symmetric and positive semidefinite by:
    
    1. Optionally symmetrizing the input matrix.
    2. Performing eigen-decomposition.
    3. Clipping eigenvalues below ``eps`` to enforce non-negativity.
    4. Reconstructing the matrix from clipped eigenvalues and eigenvectors.
    5. Symmetrizing the result again to avoid numerical drift.

    Args:
        mat (torch.Tensor): Input square matrix of shape ``(k, k)``.
        eps (float, optional): Minimum eigenvalue threshold to enforce positive definiteness.
            Defaults to ``1e-6``.
        ensure_symmetry (bool, optional): If True, symmetrize the input before decomposition.
            Defaults to False.

    Returns:
        torch.Tensor: Symmetric positive semidefinite matrix of shape ``(k, k)``.

    Example:
        >>> M = torch.tensor([[1.0, 2.0], [2.0, -3.0]])
        >>> M_psd = eigen_decomp_proj_to_pd(M)
        >>> torch.linalg.eigvalsh(M_psd)
        tensor([1.0133e-06, 1.8284e+00])
    """
    # Ensure symmetry
    if ensure_symmetry:
        mat = (mat + mat.T) / 2
    
    # Eigen-decomposition
    eigvals, eigvecs = torch.linalg.eigh(mat)
    
    # Clip eigenvalues to non-negative
    eigvals_clipped = torch.clamp(eigvals, min=eps)
    
    # Reconstruct
    mat_psd = eigvecs @ torch.diag(eigvals_clipped) @ eigvecs.T
    
    # Ensure symmetry again
    return (mat_psd + mat_psd.T) / 2

def generate_sigma_bad_and_good(
    k: int,
    proportion_var_dif: float,
    generator: torch.Generator,
    var_range: Tuple[float, float] = (0.0, 1.0),
    eps: float = 1e-6,
    device: torch.device = torch.device("cpu"),
    dtype: torch.dtype = torch.float64
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Generate a pair of covariance matrices: one 'good' baseline and one 'bad' perturbed version.

    The construction proceeds as follows:

    1. Generate two baseline covariance matrices using ``random_vcov_matrix``.
    2. Sample a random mask over the upper-triangular entries (including diagonal).
    3. Copy selected entries from the 'good' matrix into the 'bad' matrix, leaving
       others perturbed.
    4. Reflect the upper-triangular entries to the lower-triangular part to ensure symmetry.
    5. Project the 'bad' matrix onto the positive definite cone using
       :func:`eigen_decomp_proj_to_pd`.

    Args:
        k (int): Dimension of the covariance matrices.
        proportion_var_dif (float): Probability of keeping an entry different between
            the 'bad' and 'good' matrices.
        generator (torch.Generator): Random number generator for reproducibility.
        var_range (Tuple[float, float], optional): Range for diagonal variances.
            Defaults to (0.0, 1.0).
        eps (float, optional): Small diagonal perturbation to ensure positive definiteness.
            Defaults to ``1e-6``.
        device (torch.device, optional): Device for tensor allocation. Defaults to CPU.
        dtype (torch.dtype, optional): Data type of the returned tensors. Defaults to ``torch.float64``.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]:
            - ``sigma_bad``: Perturbed covariance matrix of shape ``(k, k)``, projected to PSD.
            - ``sigma_good``: Baseline covariance matrix of shape ``(k, k)``.

    Example:
        >>> g = torch.Generator().manual_seed(123)
        >>> sigma_bad, sigma_good = generate_sigma_bad_and_good(3, 0.5, generator=g)
        >>> sigma_bad.shape, sigma_good.shape
        (torch.Size([3, 3]), torch.Size([3, 3]))
    """
    # Step 1: Generate baseline matrices
    sigma_bad = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)
    sigma_good = random_vcov_matrix(k, generator=generator, var_range=var_range, device=device, dtype=dtype, eps=eps)

    # Step 2: Random mask for off-diagonal entries
    count_possible_changes = (k**2 + k) // 2 #Count diagonal entries + upper triangle
    index_change_vars = ~torch.bernoulli(torch.full((count_possible_changes,), proportion_var_dif, device=device), generator=generator).bool()

    triu_indices = torch.triu_indices(k, k, offset=0)
    indices_to_copy_sigma_bad = (triu_indices[0][index_change_vars], triu_indices[1][index_change_vars])

    sigma_good[indices_to_copy_sigma_bad] = sigma_bad[indices_to_copy_sigma_bad]
    i, j = torch.tril_indices(k, k, offset=-1)
    sigma_good[i, j] = sigma_good[j, i] # ensure symmetry
    
    sigma_good = eigen_decomp_proj_to_pd(sigma_good, eps=eps)

    return sigma_bad, sigma_good

def mvn_random_sample(
        mean : torch.Tensor, 
        cov_chol_decomp : Optional[torch.Tensor],
        n : int, 
        rng : Optional[torch.Generator] = None, 
        args_checks : bool = True
    ):
    """Generate n-vectors sampled of a multivariate normal (MVN) distribution with parameters
    mean and cov. Based on the implementation of (r)sample from 
    torch.distributions.MultivariateNormal according to torch version 2.9.1. It uses
    cholesky-decomposition method.

    Args:
        mean (torch.Tensor): Location parameter of a MVN. Shape ``(f,)`` or ``(b, f)`` or ``(1,f)``.
        cov_chol_decomp (torch.Tensor): Variance-Covariance matrix of MVN after cholesky decomposition. 
            Shape ``(f,f)``` or ``(b, f, f)`` or ``(1, f, f)``, ``cov.dim()==mean.dim()+1`` should hold.
        n (int): Count of vectors to be sampled (per batch).
        rng (Optional[torch.Generator]): If passed, sampling is done using this
            generator.
        args_checks (bool): If true, it will be checked whether the shapes of mean and
            cov are as expected, whether symmetry (w. r. t. to the last two dims for each batch)
            is given within the range of ``symmetry_rtol_atol`` for ``cov`` and type checks
            are done for ``n`` and ``rng``.`
        symmetry_rtol_atol (Tuple[float,float]): Corresponds to the (rtol, a_tol) parameters
            of ``torch.allclose``, passed as ``*args``, so ordering is important. Ignored if
            ``not args_checks``.
    Returns:
        torch.Tensor:
            A tensor of shape ``(n, f)`` or ``(n, b, f)`` containing the ``n`` sampled vectors (for each batch).

    Example:
        >>> count_covariates = 5
        >>> device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        >>> torch.set_default_dtype(torch.float32)
        >>> rng = torch.Generator(device)
        >>> mu = torch.zeros(count_covariates)
        >>> sigma_bad, _ = generate_sigma_bad_and_good(f = count_covariates, proportion_var_dif=1.0, generator = rng, device=device, dtype= torch.get_default_dtype())
        >>> sample = mvn_random_sample(mean=mu, cov=sigma_bad, n=100, rng=rng)
        >>> sample.shape
        torch.Size([100, 5])
    """
    if args_checks:
        #shape checks
        assert (mean.dim() in [1, 2, 3]) and (cov_chol_decomp.dim()==mean.dim()+1), "mean must be a single vector (rank 1 tensor) and cov a matrix (rank 2 tensor)"
        assert (mean.size(-1) == cov_chol_decomp.size(-1)) and (cov_chol_decomp.size(-1) == cov_chol_decomp.size(-2)), "mean must have shape [f] and cov shape [f, f]"
        #ensure n is an int
        n = int(n)
        assert isinstance(rng, torch.Generator) or rng is None, "rng needs to be None or a rng"

    shape = torch.Size([n]) + mean.shape
    
    eps = torch.empty(shape, dtype=mean.dtype, device = mean.device).normal_(generator=rng)

    deviations = torch.matmul(cov_chol_decomp, eps.unsqueeze(-1)).squeeze(-1) # apply decomp to each sampled vector


    return mean + deviations


# Extending CreditDataSample

Notes and experiments on how to keep tabs of the feature ids

In [5]:
import torch
from typing import Optional
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = sample = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.get_default_dtype())
)

In [17]:
nan_val = -1
inferred_labels = torch.distributions.Categorical(torch.tensor([0.8,0.1,0.1])).sample(data.features_unlabeled.shape[:-1]) - 1
mask_inferred_rej_lbls = inferred_labels != -1

self = sample
inplace : bool = True
safety_checks : bool = True

In [ ]:
from typing import Dict, Tuple, Union
def _mask2d_to_int_idxs(mask : torch.Tensor, correction_last_idx : Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
    mask_int = mask.to(torch.int32)
    idx_last_axis = torch.where(# [B, N]
            mask,
            mask_int.cumsum(-1) - 1,
            -1
    )
    if correction_last_idx is not None:
        idx_last_axis = idx_last_axis + correction_last_idx
    batch_idx = torch.arange(mask.size(0), device=mask.device).unsqueeze(-1).expand(mask.shape)
    
    return batch_idx[mask], idx_last_axis[mask]

if True:
    if True:
        if safety_checks:
            if mask_inferred_rej_lbls.dtype != torch.bool:
                raise ValueError("mask_infered_rej_lbls should be of type bool")
            if not (mask_inferred_rej_lbls.shape == inferred_labels.shape == self.features_unlabeled.shape[:-1]):
                raise ValueError("mask_inferred_rej_lbls and inferred_labels should have the same shape as self.features_unlabeled.shape[:-1]")
            
            inferred_labels_with_vals_as_saved_labels = torch.isin(inferred_labels[mask_inferred_rej_lbls].unique(), self.labels).all()
            if not inferred_labels_with_vals_as_saved_labels:
                raise ValueError("Tensor inferred_labels[mask_inferred_rej_lbls] should contain only values like in self.labels")
            
        
        *batch_shape, N_labels = self.labels.shape
        B = torch.tensor(batch_shape).prod()

        current_labels = self.labels.reshape(B, N_labels)
        mask_nans_labels = current_labels.isnan()

        N_unlabeled = mask_inferred_rej_lbls.size(-1)
        mask_inf = mask_inferred_rej_lbls.reshape(B, N_unlabeled)
        inf_lbls = inferred_labels.reshape(B, N_unlabeled)

        slots_available = mask_nans_labels.sum(dim=-1)
        slots_needed = mask_inf.sum(dim=-1)

        needed_padding = torch.maximum(slots_needed - slots_available, torch.tensor(0))
        max_needed_padding = needed_padding.max()

        # specifics
        pad_mode_val = {"mode" : 'constant', 'value':torch.nan}
        mask_valid_lbls = ~mask_nans_labels
        batch_idx_valid_lbls, N_idx_valid_lbls = _mask2d_to_int_idxs(mask_valid_lbls)
        batch_idx_inf_lbls, N_idx_inf_lbls = _mask2d_to_int_idxs(mask_inf, N_labels - slots_available.unsqueeze(-1))

        def _append_obs(append_to : torch.Tensor, to_append : torch.Tensor, pad_spec : tuple[int]):
            appended = torch.nn.functional.pad(append_to, pad=pad_spec, **pad_mode_val)
            appended[batch_idx_valid_lbls, N_idx_valid_lbls] = append_to[mask_valid_lbls].to(appended.dtype)
            appended[batch_idx_inf_lbls, N_idx_inf_lbls] = to_append[mask_inf].to(appended.dtype)

        new_labels = _append_obs(append_to=current_labels, to_append=inf_lbls, pad_spec=(0, max_needed_padding))
        new_inferred_ids = _append_obs(
            append_to=self._inferred_ids.reshape(B, N_labels), 
            to_append=self._unlabeled_ids.reshape(B, N_unlabeled),
            pad_spec=(0, max_needed_padding)
        )
        new_features_labeled = _append_obs(
            append_to=self.features_labeled.reshape(B, N_labels, -1), 
            to_append=self.features_unlabeled.reshape(B, N_unlabeled, -1),
            pad_spec=(0, 0, 0, max_needed_padding)
        )

        new_N_unlabeled = N_unlabeled - slots_needed.max()
        mask_non_inferred = ~mask_inferred_rej_lbls
        batch_idx_resized_unlbld, N_idx_resized_unlbld = _mask2d_to_int_idxs(mask_non_inferred)
        def _resize_obs(to_resize : torch.Tensor, pad_spec):
            resized = to_resize.new_full(torch.Size([B]) + to_resize.shape[2:], fill_value=torch.nan)
            resized[batch_idx_resized_unlbld, N_idx_resized_unlbld] = to_resize[mask_non_inferred]
            return resized

        features_unlabeled = _resize_obs(self.features_unlabeled)
        unlabeled_ids = _resize_obs(self._unlabeled_ids)

        new_labels, new_inferred_ids, new_features_labeled, features_unlabeled, unlabeled_ids = [
            t.reshape(torch.Size(batch_shape) + t.shape[1:]) for t in 
            new_labels, new_inferred_ids, new_features_labeled, features_unlabeled, unlabeled_ids
        ]
            
        if inplace:
            self.features_labeled = new_features_labeled
            self.labels = new_labels
            self._inferred_ids = new_inferred_ids

            self.features_unlabeled = features_unlabeled
            self._unlabeled_ids = unlabeled_ids
            #return
        new_instance = CreditDataSample(
            features_rejects=features_unlabeled,
            features_accepts=new_features_labeled,
            default_flag_accepts=new_labels,
            ids_rejects=unlabeled_ids
        )
        new_instance._inferred_ids = new_inferred_ids
        #return new_instance

                                                                    
nan_val = -1
inferred_labels = torch.distributions.Categorical(torch.tensor([0.8,0.1,0.1])).sample(data.features_unlabeled.shape[:-1]) - 1
mask_inferred_rej_lbls = inferred_labels != -1


## Creating clone method (is_valid_clone already created)

In [ ]:
import torch
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.int8)
)

In [4]:
from typing import Optional

if True:
    def new_instance_with_full_info(
            features_unlabeled : torch.Tensor,
            unlabeled_ids : torch.Tensor,
            features_labeled : torch.Tensor,
            labels : torch.tensor,
            ids_inferred : torch.Tensor,
            nan_val_ids_rejects : torch.Tensor,
            nan_val_labels : torch.Tensor,
            retrieve_only_labeled : bool = True,
            seed : Optional[int] = None,
            rng_state : Optional[torch.Tensor] = None,
            safety_checks = True
    ):
        new_instance = CreditDataSample(
            features_rejects=features_unlabeled,
            features_accepts=features_labeled,
            default_flag_accepts=labels,
            ids_rejects=unlabeled_ids,
            retrieve_only_labeled=retrieve_only_labeled,
            seed=seed,
            safety_checks=safety_checks,
            nan_value_labels=nan_val_labels,
            nan_value_ids_rejects=nan_val_ids_rejects
        )
        if safety_checks:
            if ids_inferred.shape != new_instance.labels.shape:
                raise ValueError("ids_inferred should have the same shape as labels (after init)")
            mask_id_rej_is_nan = new_instance._ids_rejects_nan_checker(ids_inferred)
            mask_labels_is_nan = new_instance._labels_nan_checker(new_instance.labels)

            any_label_is_nan_and_id_is_not = torch.any(mask_id_rej_is_nan < mask_labels_is_nan)
            if any_label_is_nan_and_id_is_not:
                raise ValueError("NaNs pattern in ids_inferred not compatible with labels NaN pattern")
            
        new_instance._ids_inferred = ids_inferred

        if rng_state is not None:
            new_instance.rng.set_state(rng_state)

        return new_instance


    #CreditDataSample.new_instance_with_full_info = new_instance_with_full_info
    def clone(self, safety_data_integrety_tests : bool = False):
        if safety_data_integrety_tests:
            for ten_name in CreditDataSample._tensor_attr_after_init:
                if not isinstance(getattr(self, ten_name, None), torch.Tensor):
                    raise ValueError(f"Member {ten_name} was non existent or not a torch.Tensor")
                
        kwargs_new_instance_with_full_info = {
            (k[1:] if k[0]=='_' else k) : getattr(self, k).clone() for k in CreditDataSample._tensor_attr_after_init
        }
        
        return CreditDataSample.new_instance_with_full_info(
            retrieve_only_labeled = self.retrieve_only_labeled,
            rng_state = self.rng.get_state(),
            safety_checks=safety_data_integrety_tests,
            **kwargs_new_instance_with_full_info
        )
    
data.is_valid_clone(clone(data, True))

(True, '')

### Develop train test split being aware of the nan structure

Now in the batched situation we have the problem that there are different amount of valid entries per batch. 
This needs to be considered to avoid having one of them having all the nans.

Specially important is to still have a normalized form at the end. Therefore we need to also pass to 
`_generate_random_train_test_idxs` a mask containing the nan positions. This mask will be of course boolean
and its dimensions would be $B_1, \ldots, B_{r-1}, N$ with $r \geq 2$ being the tensor-rank of the mask. Without 
loss of generality let's call $B = \prod_{i=1}^{r-1} B_i \in \mathbb{N}$ the flattened batch dimensions size.
Let then $M \in \{0,1\}^{B \times N}$ be the boolean mask with flattened batch dimensions, where $1$ represents an
entry being unvalid/`nan`. So the problem could be stated as following:

Let $(v_b)_{b\in \{1, \ldots B\}}$ with $0 \leq v_b = N - \sum_{n=1}^N M[b, n] \leq N$ the finite series containing the amount of
valid entries per batch. Also let $\rho_t \in (0,1)$ be the test-proportion. Also let $s_b := \operatorname{round}(\rho_t \cdot v_b)$
be the test size per batch with $\operatorname{round}$ being the bankers rounding. Generating a gather ready index of size $s_b$
selecting distinct valid entries is trivial. The question however is if after selecting $s_b$ elements we would still have enough
unvalid entries to use them to end up generating a gather ready index that would select in a normalized manner the desired valid
entries and use **already existing** `nan` entries to avoid the overhead of padding or scattering in a full `nan` tensor.

The natural shape for such index would be `[B, S]` with 
$$
\texttt{S} := \operatorname{round}\left(\rho \cdot \max_{b \in \{1, \ldots B\}} v_b \right) =: \operatorname{round}(\rho \cdot V)
$$.
To prove whether there would be enough unvalid indices left per batch to select from, so that the resulting gather ready index
would have the desired normalized shape it suffices to prove

$$ 
\forall b \in \{1, \ldots B \} : \underbrace{\operatorname{round}(V \cdot \rho )}_{\hat{=} \texttt{S}}
 - \underbrace{\operatorname{round}(v_b \cdot \rho)}_{\hat{=} s_b} 
 \leq \underbrace{N - v_b = \sum_{n=1}^N M[b, n]}_{\text{Left unvalid entries}}
$$

The proof goes as follows (**disclaimer:** The problem was stated by the author, the formal proof was AI generated
by `ChatGPT` see [here](https://chatgpt.com/share/69872f18-124c-8008-b645-dc2cf54764f5) and 
[here](https://chatgpt.com/share/69874c4c-c3f8-8008-967b-5b31b597956b) for the link containing the chat)

#### Proof

First note that

$$
\forall x \in \mathbb{R} : x - \frac{1}{2} \le \operatorname{round}(x) \le x + \frac{1}{2}.
$$

Therefore
$$
\Rightarrow \operatorname{round}(\rho \cdot V) \leq \rho V + \frac{1}{2}, \quad \operatorname{round}(\rho v_b) \geq \rho v_b - \frac{1}{2}
$$

and

$$
\begin{aligned}
\Rightarrow \operatorname{round}(\rho \cdot V) - \operatorname{round}(\rho v_b) &\leq \rho \left(V - v_b \right) + 1 \\
& \overset{\rho < 1}{\leq} V - v_b + 1
\end{aligned}
$$

from which the proposition follows:

$$
\begin{aligned}
\Leftrightarrow \operatorname{round}(\rho \cdot V) - \operatorname{round}(\rho v_b) &\leq V - v_b \\
& \overset{V \le N}{\leq} N - v_b \quad \quad \square
\end{aligned}
$$



In [1]:
import torch
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.int8)
)

In [ ]:
from typing import Union, Tuple
def _generate_random_train_test_idxs(
    self,
    shape_up_to_N_dim: Union[tuple[int], torch.Size],
    test_proportion: float,
    nan_mask: torch.Tensor,  # [..., N]
) -> Tuple[torch.Tensor, torch.Tensor]:
    r"""
    Generate random train/test index splits along the sample dimension,
    selecting only from valid (non-NaN) positions, while ensuring that the
    resulting index tensors have *normalized shapes* across all super-batch
    slices.

    This method supports super-batch inputs of shape ``[..., N]`` where each
    slice may contain a different number of valid (non-NaN) positions. The
    split is performed **per slice**, based only on the number of valid
    positions in that slice.

    The key property of this method is that the returned index tensors have
    shapes::

        train_indices: [..., max_train_count]
        test_indices:  [..., max_test_count]

    where ``max_train_count`` and ``max_test_count`` are the maximum numbers
    of train/test samples required by any slice in the super-batch.

    To achieve this *normalized* shape, the method proceeds in two stages:

    1. **Valid selection**  
       For each slice, the valid positions are randomly permuted and the
       first ``test_count[b]`` (resp. ``train_count[b]``) positions are
       selected as the true test/train indices.

    2. **NaN-based padding**  
       If a slice has fewer valid positions than the global maximum
       ``max_test_count`` (resp. ``max_train_count``), the remaining slots
       are filled by selecting indices corresponding to invalid (NaN)
       positions. These NaN positions are taken from the end of the sorted
       index list, ensuring that:
       
       - valid positions are always chosen first,
       - NaN positions are only used as padding,
       - the final shapes are rectangular and suitable for ``gather``.

    This guarantees that all returned index tensors are gather-compatible
    and have consistent shapes across the entire super-batch, even when the
    number of valid samples varies per slice.

    Args:
        shape_up_to_N_dim (tuple or torch.Size):
            Super-batch shape ending with ``N``.
        test_proportion (float):
            Fraction of *valid* samples to assign to the test set.
        nan_mask (Tensor):
            Boolean mask of invalid positions, shape ``[..., N]``.

    Returns:
        (Tensor, Tensor):
            ``(train_indices, test_indices)``, each of shape
            ``[..., max_train_count]`` and ``[..., max_test_count]``,
            containing valid indices first and NaN-padding indices last.
    """
    # Mask of valid positions
    valid_mask = ~nan_mask  # [..., N]

    # Count valid positions per super-batch slice
    valid_counts = valid_mask.sum(dim=-1, keepdim=True)  # [..., 1]

    # Compute per-slice test counts (rounding per slice)
    test_counts = (valid_counts * test_proportion).round().to(torch.long)  # [..., 1]

    # Random scores only for valid positions
    scores = torch.where(
        valid_mask,
        torch.rand(shape_up_to_N_dim, generator=self.rng, device=self.device),
        float('inf')
    )

    # Sort so valid positions come first
    scores_idx = scores.argsort(dim=-1)  # [..., N]

    # N_arange mask to get valid scores_ids
    N = shape_up_to_N_dim[-1]
    arange_N = torch.arange(N, device=scores_idx.device).view( # [1,...,1, N]
        *([1]*(scores_idx.ndim-1)), N
    )

    # Mask for selecting test indices
    ## Get indices among valid positions corresponding exactly
    ## to the amount of needed tests
    mask_needed_tests = arange_N < test_counts # [..., N]
    ## Get indices for appending unvalid positions to allow
    ## for normalized shapes (see proof to check that it is assured
    ## there are still enough)
    max_test_count = test_counts.max()
    counts_to_append_test = max_test_count - test_counts # [..., N]
    mask_unvalids_to_append_to_tests = arange_N >= (N-counts_to_append_test) # [..., N]
    ## Get the mask through both
    test_mask = mask_needed_tests | mask_unvalids_to_append_to_tests # [..., N]

    # Mask for selecting train_indices (same logic as test)
    ## Positions k with test_count[b] <= pos[b] < valid_counts[b]
    mask_needed_trains = ~test_mask & (arange_N < valid_counts)
    ## For appending
    train_counts = valid_counts - test_counts # [..., 1]
    max_train_count = train_counts.max()
    counts_to_append_train = max_train_count - train_counts
    mask_unvalids_to_append_to_trains =  arange_N >= (N-counts_to_append_train) # [..., N]
    ## mask
    train_mask = mask_needed_trains | mask_unvalids_to_append_to_trains


    # Gather the indices
    batch_dims = shape_up_to_N_dim[:-1]
    test_indices = scores_idx.masked_select(test_mask).reshape(*batch_dims, max_test_count)
    train_indices = scores_idx.masked_select(train_mask).reshape(*batch_dims, max_train_count)

    return train_indices, test_indices

train_indices, test_indices  = _generate_random_train_test_idxs(
    data,
    data.labels.shape,
    0.2,
    data._labels_nan_checker(data.labels) # Private api - only for testing purposes here
)
train_indices.shape, train_indices.shape[:-1] == data.super_batch_shape

(torch.Size([32, 160]), True)

In [5]:
if True:
      def train_test_split(self, test_proportion: float, check_data_integrity_before_returning : bool = False):
        """
        Split the dataset into train and test subsets without leakage.

        The split is performed independently for labeled and unlabeled pools,
        preserving super-batch structure and using gather-based indexing.

        Args:
            test_proportion (float):
                Fraction of samples to assign to the test set.
            check_data_integrity_before_returning (bool):
                Whether to check if all data in the class fullfills the expected
                characteristics. Only necessary/sensible if the tensors in the class
                have been changed manually, like adding more data - which is not the
                intentede purpose of the clase. Defaults to ``False``.

        Returns:
            (CreditDataSample, CreditDataSample):
                (``train_sample``, ``test_sample``), each containing consistent subsets of
                features, labels, IDs, and inferred-ID tracking.
        """
        if not (0.0 <= test_proportion <= 1.0):
            raise ValueError("test_proportion must be between 0 and 1")

        # Generate masks
        gather_idx_train_unlbld, gather_idx_test_unlbld = self._generate_random_train_test_idxs(
            shape_up_to_N_dim=self._unlabeled_ids.shape,
            test_proportion=test_proportion,
            nan_mask=self._ids_rejects_nan_checker(self._unlabeled_ids)
        )
        gather_idx_train_lbld, gather_idx_test_lbld = self._generate_random_train_test_idxs(
            shape_up_to_N_dim=self.labels.shape, 
            test_proportion=test_proportion,
            nan_mask=self._labels_nan_checker(self.labels)
        )

        gather_features = lambda gather_from, idx_gather : gather_from.gather(
            dim=-2,
            index=idx_gather.unsqueeze(-1).expand(*idx_gather.shape, self.features_count)
        )

        shared_args_for_new_instances = lambda : {
            "retrieve_only_labeled" : self.retrieve_only_labeled,
            "nan_val_ids_rejects" : self._nan_val_ids_rejects.clone(),
            "nan_val_labels" : self._nan_val_labels.clone(),
            "rng_state" : self.rng.get_state()
        }

        # Slice data
        train_sample = CreditDataSample.new_instance_with_full_info(
            features_unlabeled =    gather_features(self.features_unlabeled, gather_idx_train_unlbld),
            unlabeled_ids =         self._unlabeled_ids.gather(dim=-1, index=gather_idx_train_unlbld),
            features_labeled =      gather_features(self.features_labeled, gather_idx_train_lbld),
            labels =                self.labels.gather(dim=-1, index=gather_idx_train_lbld),
            ids_inferred =          self._ids_inferred.gather(dim=-1, index=gather_idx_train_lbld),
            safety_checks =         check_data_integrity_before_returning,
            **shared_args_for_new_instances()
        )

        test_sample = CreditDataSample.new_instance_with_full_info(
            features_unlabeled =    gather_features(self.features_unlabeled, gather_idx_test_unlbld),
            unlabeled_ids =         self._unlabeled_ids.gather(dim=-1, index=gather_idx_test_unlbld),
            features_labeled =      gather_features(self.features_labeled, gather_idx_test_lbld),
            labels =                self.labels.gather(dim=-1, index=gather_idx_test_lbld),
            ids_inferred =          self._ids_inferred.gather(dim=-1, index=gather_idx_test_lbld),
            safety_checks =         check_data_integrity_before_returning,
            **shared_args_for_new_instances()
        )

        return train_sample, test_sample
      
train_test_split(data, test_proportion=0.5, check_data_integrity_before_returning=True)

(<berebasl.simulation.credit_data_simulation.CreditDataSample at 0x198bb60e9c0>,
 <berebasl.simulation.credit_data_simulation.CreditDataSample at 0x198bcf6fe50>)

### Random permutations for `__getitem__`

A final step is to change the logic for `__getitem__`. Right now it takes an observation per super-batch of the exact same
position as in all other super batches. This is not sensible in ML in general. So a random permutation makes more sense.

In [20]:
import torch
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = CreditDataSample(
    features_rejects=torch.randn(8,B, N_r, C),
    features_accepts=torch.randn(8,B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (8,B, N_a), dtype=torch.int8)
)

In [ ]:
def to_append_to_init(self):
    self._random_perm_idx_labeled = torch.rand_like(self.labels, dtype=torch.float32).argsort(dim=-1)
    self._random_perm_idx_unlabeled = torch.rand_like(self._unlabeled_ids, dtype=torch.float32).argsort(dim=-1)

def __getitem__(self, idx:int):
    if self.retrieve_only_labeled:
        random_idxs = self._random_perm_idx_labeled[..., idx].unsqueeze(-1)
        return {
            "features": self.features_labeled.gather(
                dim=-2,
                index=random_idxs.unsqueeze(-1).expand(*random_idxs.shape, self.features_count)
            ).squeeze(-2),
            "default_flag": self.labels.gather(dim=-1, index=random_idxs).squeeze(-1),
            "is_inferred" : self.mask_inferred_lbls.gather(dim=-1, index=random_idxs).squeeze(-1),
            "accepted": True,
        }
    random_idxs = self._random_perm_idx_unlabeled[..., idx].unsqueeze(-1)
    return {
        "features": self.features_unlabeled.gather(
            dim=-2,
            index=random_idxs.unsqueeze(-1).expand(*random_idxs.shape, self.features_count)
        ).squeeze(-2),
        "default_flag": None,
        "is_inferred" : None,
        "accepted": False,
    }

data.retrieve_only_labeled = True

to_append_to_init(data)
item = __getitem__(data, torch.randint(len(data), size=(1,)).item())

### Utility for inspecting valid labeled and unlabeled

In [33]:
import torch
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


B, N_r, N_a, C = 32, 500, 200, 5

data = CreditDataSample(
    features_rejects=torch.randn(B, N_r, C),
    features_accepts=torch.randn(B, N_a, C),
    default_flag_accepts=torch.randint(0,2, (B, N_a), dtype=torch.int8)
)

In [48]:
import pandas as pd
from IPython.display import display

def inspect_data(self, as_pd_ready_dicts: bool = False):
    r"""
    Inspect labeled and unlabeled data after applying validity masks for IDs, labels,
    and features. The method returns two dictionaries containing the surviving
    observations in either tensor-structured form or pandas-ready columnar form.

    Args:
        as_pd_ready_dicts (bool, optional):
            If ``True``, the returned dictionaries map column names to 1-D tensors,
            suitable for direct construction of a ``pandas.DataFrame``.
            If ``False``, the returned dictionaries group tensors by semantic role
            (``"batch_iders"``, ``"features"``, ``"label"``, etc.).
            Defaults to ``False``.

    Returns:
        tuple[dict, dict]:
            A pair ``(unlabeled_data, labeled_data)`` where each element is a dictionary
            describing the valid observations after masking. Let ``N_vu`` be the amount
            of valid unlabeled observations and ``N_vl`` the amount of valid labeled
            observations. Further ``B=len(self.super_batch_shape)`` and 
            ``F=self.features_count``

            **Unlabeled data fields (``as_pd_ready_dicts=False``):**
            - ``"batch_iders"`` — integer tensor of shape ``(N_vu, B)`` giving the
              super-batch coordinates of each valid unlabeled observation.
            - ``"ids"`` — tensor of shape ``(N_vu,)`` containing valid observation IDs.
            - ``"features"`` — tensor of shape ``(N_vu, F)`` containing feature vectors.

            **Labeled data fields (``as_pd_ready_dicts=False``):**
            - ``"batch_iders"`` — integer tensor of shape ``(N_vl, B)`` giving the
              super-batch coordinates of each valid labeled observation.
            - ``"features"`` — tensor of shape ``(N_vl, F)`` containing feature vectors.
            - ``"label"`` — tensor of shape ``(N_vl,)`` with valid labels.
            - ``"label_is_inferred"`` — boolean mask of shape ``(N_vl,)`` indicating
              whether each label was inferred rather than observed.
            - ``"inferred_obs_id"`` — tensor of shape ``(N_vl,)`` containing the inferred
              observation IDs used for reject-inference logic.

            **Pandas-ready mode (``as_pd_ready_dicts=True``):**
            Each batch dimension ``b`` is exported as a separate column
            ``"super_batch_dim:{b}"`` with shape ``(N,)``.
            Each feature dimension ``f`` is exported as ``"F{f}"``.
            All other fields (IDs, labels, inferred IDs) are exported as 1-D tensors.

    Notes:
        The validity masks are computed using the class-specific ID and label
        NaN-checkers. 

        Super-batch coordinates are obtained via ``torch.where`` and stacked along the
        last dimension, yielding a tensor of shape ``(N, B)`` where ``B`` is the number
        of super-batch dimensions.

    """
    mask_valid_unlabeled = ~self._ids_rejects_nan_checker(self._unlabeled_ids)
    batch_iders_unlabeled = torch.stack(
        torch.where(mask_valid_unlabeled),
        dim=-1
    )
    valid_features_unlabeled = self.features_unlabeled[mask_valid_unlabeled]
    valid_ids_unlabeled = self._unlabeled_ids[mask_valid_unlabeled]

    mask_valid_labeled =  ~self._labels_nan_checker(self.labels)
    batch_iders_labeled = torch.stack(
        torch.where(mask_valid_labeled),
        dim=-1
    )
    valid_features_labeled = self.features_labeled[mask_valid_labeled]
    valid_labels = self.labels[mask_valid_labeled]
    valid_inferred_ids = self._ids_inferred[mask_valid_labeled]
    valid_label_is_inferred = ~self._ids_rejects_nan_checker(valid_inferred_ids)

    if as_pd_ready_dicts:
        batch_id_dict_maker = lambda iders : {f"super_batch_dim:{b_idx}" : iders[:,b_idx] for b_idx in range(len(self.super_batch_shape))}
        feats_dict_maker = lambda feats : {f"F{f_idx}" : feats[:, f_idx] for f_idx in range(self.features_count)}

        batch_iders_unlbld_dict = batch_id_dict_maker(batch_iders_unlabeled)
        feats_unlbld_dict = feats_dict_maker(valid_features_unlabeled)
        ids_unlbld_dict = {"obs_id" : valid_ids_unlabeled}

        unlabeled_data = batch_iders_unlbld_dict | ids_unlbld_dict | feats_unlbld_dict

        batch_iders_lbld_dict = batch_id_dict_maker(batch_iders_labeled)
        feats_lbld_dict = feats_dict_maker(valid_features_labeled)
        other_lbld_attr = {
            "label" : valid_labels,
            "label_is_inferred" : valid_label_is_inferred,
            "inferred_obs_id" : valid_inferred_ids
        }
        labeled_data = batch_iders_lbld_dict | feats_lbld_dict | other_lbld_attr

    else: 
        unlabeled_data = {
            "batch_iders" : batch_iders_unlabeled,
            "ids" : valid_ids_unlabeled,
            "features" : valid_features_unlabeled
        }
        labeled_data = {
            "batch_iders" : batch_iders_labeled,
            "features" : valid_features_labeled,
            "label" : valid_labels,
            "label_is_inferred" : valid_label_is_inferred,
            "inferred_obs_id" : valid_inferred_ids
        }
        
    return unlabeled_data, labeled_data

non_pd_ready = inspect_data(data, as_pd_ready_dicts=False)
print("**Non-pd.DataFrame ready**")
print("unlabeled_data keys:")
print(non_pd_ready[0].keys())
print("labeled_data keys:")
print(non_pd_ready[1].keys())

pd_ready = inspect_data(data, as_pd_ready_dicts=True)
print("**As pd.DataFrame**")
print("Labeled:")
display(pd.DataFrame(pd_ready[0]))
print("Unlabeled:")
display(pd.DataFrame(pd_ready[1]))

**Non-pd.DataFrame ready**
unlabeled_data keys:
dict_keys(['batch_iders', 'ids', 'features'])
labeled_data keys:
dict_keys(['batch_iders', 'features', 'label', 'label_is_inferred', 'inferred_obs_id'])
**As pd.DataFrame**
Labeled:


,super_batch_dim:0,obs_id,F0,F1,F2,F3,F4
0,0,0,-0.755405,0.816141,0.277253,1.376728,1.804941
1,0,1,-0.219928,0.126998,0.340908,1.144932,0.278768
2,0,2,0.341845,1.171781,0.031336,-0.933281,-0.018658
3,0,3,0.822129,1.550410,1.180782,-0.240260,1.101663
4,0,4,0.129543,0.953418,-1.412877,-0.809141,-0.297054
...,...,...,...,...,...,...,...
15995,31,15995,0.322797,1.646764,0.306727,1.475605,1.132182
15996,31,15996,1.522892,-1.279827,-0.501342,0.979944,-0.794504
15997,31,15997,0.889031,0.146948,1.744296,0.768771,-0.878495
15998,31,15998,-0.273977,1.347947,-1.393794,0.191830,0.624252


Unlabeled:


,super_batch_dim:0,F0,F1,F2,F3,F4,label,label_is_inferred,inferred_obs_id
0,0,-1.355426,-0.639931,-0.284010,0.539957,-2.323570,0,False,-1
1,0,-0.250032,0.974328,0.515574,-0.419847,-0.778623,1,False,-1
2,0,0.774754,-0.432358,-0.694887,0.866955,0.429485,0,False,-1
3,0,-0.172295,1.148865,-0.052102,0.188453,-0.021585,1,False,-1
4,0,0.813102,0.524723,1.563170,-0.171042,-2.027699,1,False,-1
...,...,...,...,...,...,...,...,...,...
6395,31,0.195881,0.673826,0.203161,-0.865566,0.427558,0,False,-1
6396,31,0.579732,1.480196,0.396095,-0.310245,-0.841653,1,False,-1
6397,31,0.103250,1.026619,0.587010,0.408940,-1.940030,0,False,-1
6398,31,1.412066,0.469856,-0.942877,2.008236,0.795386,1,False,-1


## Make sure things work as expected

In [9]:
import torch
import os
import sys
base_path = os.path.abspath("./..")
if base_path not in sys.path:
    sys.path.append(base_path)

from berebasl.simulation.credit_data_simulation import CreditDataSample


super_b, B, N_r, N_a, C = 8, 32, 500, 200, 5

def list_available_devices():
    devices = ["cpu"]

    # CUDA (NVIDIA)
    if torch.cuda.is_available():
        devices += [f"cuda:{i}" for i in range(torch.cuda.device_count())]

    # MPS (Apple Silicon)
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        devices.append("mps")

    return devices

In [14]:
for device in list_available_devices():
    print("Testing device:", device)
    device = torch.device(device)
    for batch_form in [(super_b, B),(B,), tuple([])]:
        print("Beginning testing for batch-form:", batch_form)
        data = CreditDataSample(
            features_rejects=torch.randn(*batch_form, N_r, C, device=device),
            features_accepts=torch.randn(*batch_form, N_a, C, device=device),
            default_flag_accepts=torch.randint(0,2, batch_form + (N_a,), dtype=torch.int8, device=device)
        )

        cloning_succesful, error_msg = data.is_valid_clone(data.clone(safety_data_integrety_tests=True))

        if not cloning_succesful:
            raise ValueError(error_msg)

        idx_labeling = 0

        while idx_labeling < 25 and (~data._ids_rejects_nan_checker(data._unlabeled_ids)).sum() > 0:
            idx_labeling += 1
            probs_inferred_lables = torch.where(
                data._ids_rejects_nan_checker(data._unlabeled_ids).unsqueeze(-1).expand(*data._unlabeled_ids.shape, 3),
                torch.tensor([1.0,0.0,0.0]).expand(*data._unlabeled_ids.shape, 3),
                torch.tensor([0.8,0.1,0.1]).expand(*data._unlabeled_ids.shape, 3)
            )
            inferred_labels = torch.distributions.Categorical(probs_inferred_lables).sample() - 1
            mask_inferred_rej_lbls = inferred_labels != -1

            # Not doing inplace as with safety checks all shape and nans-checks are done.
            data = data.label_rejects(inferred_labels, mask_inferred_rej_lbls, inplace=False, safety_checks=True)

            train_data, test_data = data.train_test_split(
                test_proportion=0.5,
                check_data_integrity_before_returning=True
            )

            if not (train_data.rng.get_state() == test_data.rng.get_state()).all():
                raise ValueError("Rng states is not equal after split")

            if not mask_inferred_rej_lbls.any():
                break
        print("\tBatch-form worked. Labeling rounds:", idx_labeling)

Testing device: cpu
Beginning testing for batch-form: (8, 32)
	Batch-form worked. Labeling rounds: 25
Beginning testing for batch-form: (32,)
	Batch-form worked. Labeling rounds: 25
Beginning testing for batch-form: ()
	Batch-form worked. Labeling rounds: 22
